# Different Ways to Chunk Podcast and PDF

Step 1 setup notebook for the lab.

This notebook starts by installing the required packages, loading environment variables, and preparing the two source files:
- the PDF document
- the podcast audio file

The podcast transcription is saved to a local text file so later steps can reuse it without re-transcribing.

## Step 1 - Setup and data loading

Run the cells below in order. If your files live in different locations, update the path variables in the configuration cell.

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
env_path = PROJECT_ROOT / ".env"

print("Current working directory:", PROJECT_ROOT)
print(".env exists:", env_path.exists())
print(".env path:", env_path)

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")
enable_transcription = os.getenv("ENABLE_TRANSCRIPTION")

print("OPENAI_API_KEY exists:", api_key is not None)
print("OPENAI_API_KEY preview:", api_key[:7] + "..." if api_key else None)
print("ENABLE_TRANSCRIPTION:", enable_transcription)

Current working directory: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF
.env exists: True
.env path: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF\.env
OPENAI_API_KEY exists: True
OPENAI_API_KEY preview: sk-svca...
ENABLE_TRANSCRIPTION: 1


In [2]:
%pip install -q langchain langchain-community pypdf2 python-dotenv openai tiktoken

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os

from dotenv import load_dotenv
from PyPDF2 import PdfReader

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

PROJECT_ROOT = Path.cwd()

load_dotenv(dotenv_path=PROJECT_ROOT / ".env", override=True)

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Update these paths if your files are stored somewhere else.
PDF_PATH = Path(os.getenv("PDF_PATH", DATA_DIR / "trustworthy_ai.pdf"))
AUDIO_PATH = Path(os.getenv("PODCAST_AUDIO_PATH", DATA_DIR / "trustworthy_ai_podcast.m4a"))
TRANSCRIPT_PATH = Path(os.getenv("PODCAST_TRANSCRIPT_PATH", OUTPUT_DIR / "podcast_transcript.txt"))

OPENAI_TRANSCRIPTION_MODEL = os.getenv("OPENAI_TRANSCRIPTION_MODEL", "whisper-1")
ENABLE_TRANSCRIPTION = os.getenv("ENABLE_TRANSCRIPTION", "0").lower() in {"1", "true", "yes"}

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF path: {PDF_PATH}")
print(f"Audio path: {AUDIO_PATH}")
print(f"Transcript path: {TRANSCRIPT_PATH}")
print(f"Enable transcription: {ENABLE_TRANSCRIPTION}")

Project root: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF
PDF path: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF\data\trustworthy_ai.pdf
Audio path: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF\data\trustworthy_ai_podcast.m4a
Transcript path: c:\Users\NunoAlmeida\OneDrive - Transparity\Documents\Pessoal\IronHack\LAB  Different Ways to Chunk Podcast and PDF\outputs\podcast_transcript.txt
Enable transcription: True


In [4]:
def load_pdf_text(pdf_path: Path):
    """Return the full PDF text and one entry per page with page metadata."""
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    reader = PdfReader(str(pdf_path))
    page_texts = []
    all_text = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        page_texts.append({
            "page_number": page_number,
            "text": page_text,
        })
        all_text.append(page_text)

    return "\n\n".join(all_text), page_texts


def load_podcast_transcript(transcript_path: Path):
    """Load an existing transcript file if it exists."""
    if transcript_path.exists() and transcript_path.stat().st_size > 0:
        return transcript_path.read_text(encoding="utf-8")
    return None

In [5]:
def transcribe_audio(audio_path: Path, transcript_path: Path, *, allow_skip: bool = True):
    if transcript_path.exists() and transcript_path.stat().st_size > 0:
        return transcript_path.read_text(encoding="utf-8")

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        if allow_skip:
            return None
        raise EnvironmentError("OPENAI_API_KEY is not set.")

    if OpenAI is None:
        raise ImportError("openai package is not available. Re-run the install cell.")

    max_upload_bytes = 25 * 1024 * 1024
    audio_size = audio_path.stat().st_size
    if audio_size > max_upload_bytes:
        message = (
            f"Audio file is {audio_size} bytes, which is above the 25 MB upload limit for OpenAI speech-to-text. "
            "Split the audio into smaller chunks or compress it, then rerun transcription."
        )
        if allow_skip:
            print(message)
            return None
        raise ValueError(message)

    client = OpenAI(api_key=api_key)
    try:
        with audio_path.open("rb") as audio_file:
            result = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
            )
    except Exception as exc:
        message = str(exc)
        if "413" in message or "Maximum content size limit" in message:
            if allow_skip:
                print("Transcription skipped because the audio exceeds the API upload limit.")
                return None
            raise ValueError("Transcription skipped because the audio exceeds the API upload limit.") from exc
        raise

    transcript_text = result.text
    transcript_path.write_text(transcript_text, encoding="utf-8")
    return transcript_text


def load_or_transcribe_podcast(audio_path: Path, transcript_path: Path, *, allow_skip: bool = True):
    return transcribe_audio(audio_path, transcript_path, allow_skip=allow_skip)


In [6]:
# Load the PDF text.
pdf_text = None
pdf_pages = []
if PDF_PATH.exists():
    pdf_text, pdf_pages = load_pdf_text(PDF_PATH)
    print(f"Loaded PDF pages: {len(pdf_pages)}")
    print(f"PDF text characters: {len(pdf_text)}")
    print("PDF preview:")
    print(pdf_text[:1000])
else:
    print("PDF file not found yet. Place it at the path shown above or update PDF_PATH.")

# Load or create the podcast transcript.
podcast_text = None
if TRANSCRIPT_PATH.exists() and TRANSCRIPT_PATH.stat().st_size > 0:
    podcast_text = TRANSCRIPT_PATH.read_text(encoding="utf-8")
    print(f"Loaded existing transcript: {TRANSCRIPT_PATH}")
    print(f"Transcript characters: {len(podcast_text)}")
    print("Transcript preview:")
    print(podcast_text[:1000])
elif AUDIO_PATH.exists() and ENABLE_TRANSCRIPTION:
    print("Transcript file not found, so the audio will be transcribed now.")
    podcast_text = load_or_transcribe_podcast(AUDIO_PATH, TRANSCRIPT_PATH, allow_skip=True)
    if podcast_text:
        print(f"Transcript saved to: {TRANSCRIPT_PATH}")
        print(f"Transcript characters: {len(podcast_text)}")
        print("Transcript preview:")
        print(podcast_text[:1000])
    else:
        print("Transcription was skipped because the audio is too large or could not be processed.")
        print("Provide a smaller audio file or drop a text transcript at the transcript path.")
elif AUDIO_PATH.exists():
    print("Podcast audio is present, but transcription is disabled for now.")
    print("Set ENABLE_TRANSCRIPTION=1 and OPENAI_API_KEY to generate the transcript, or place a text transcript at the transcript path.")
else:
    print("Podcast audio file not found yet. Place it at the path shown above or update AUDIO_PATH.")

Loaded PDF pages: 41
PDF text characters: 158118
PDF preview:
 
  
 
 
 
INDEPENDENT  
HIGH-LEVEL EXPERT GROUP ON  
ARTIFICIAL INTELLIGENCE  
SET UP BY THE EUROPEAN COMMISSION  
 
 
 
 
 
 
ETHICS GUIDELINES  
FOR TRUSTWORTHY AI 
 
 

 
  
 
ETHICS GUIDELINES  FOR  TRUSTWORTHY  AI 
 
High -Level Expert Group on Artificial Intelligence  
 
 
 
 
 
 
 
 
This document was written by the High -Level Expert Group on AI (AI HLEG) . The members of the AI HLEG 
named in this document support the overall framework for Trustworthy  AI put forward in these Guidelines, 
although they do not necessarily agree with every single statement in the document .  
 
The Trustworthy  AI assessment  list presented in Chapter III of this document will undergo a piloting phase by 
stakeholders to gather practical feedback. A revised version  of the assessment  list, taking into account the 
feedback gathered through the piloting phase, will be presented to the Eu ropean Commission in early 2020.  
 
 
The AI 

## What this step gives you

- A notebook ready for the lab
- A reusable PDF loader
- A podcast transcription flow that saves a local text file
- A quick preview so you can verify the inputs before chunking

Next step: chunk the two texts in different ways and compare the results.

## Step 2 - Fixed-size chunking

This step splits the PDF and podcast text into fixed-size pieces so you can see how much context gets cut off at the boundaries.

The point here is not to find the best setting immediately. The point is to create a baseline that we can compare against the smarter splitters later.

In [7]:
from langchain_text_splitters import CharacterTextSplitter


def build_fixed_splitter(chunk_size: int, chunk_overlap: int) -> CharacterTextSplitter:
    return CharacterTextSplitter(
        separator="",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )


def summarize_chunks(chunks, label: str, chunk_size: int, chunk_overlap: int):
    lengths = [len(chunk) for chunk in chunks]
    if not lengths:
        print(f"{label}: no chunks produced")
        return []

    print(f"{label} | chunk_size={chunk_size} | overlap={chunk_overlap}")
    print(f"  chunks: {len(chunks)}")
    print(f"  min chars: {min(lengths)}")
    print(f"  max chars: {max(lengths)}")
    print(f"  avg chars: {sum(lengths) / len(lengths):.1f}")
    print(f"  first chunk preview: {chunks[0][:200].replace(chr(10), ' ')}")
    print(f"  last chunk preview: {chunks[-1][-200:].replace(chr(10), ' ')}")
    return [{
        "label": label,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "chunk_index": i + 1,
        "chunk_chars": len(chunk),
        "chunk_text": chunk,
    } for i, chunk in enumerate(chunks)]


def run_fixed_size_experiments(text: str, label: str, sizes=(500, 1000, 2000), overlaps=(0, 50, 100)):
    if not text:
        print(f"{label}: no text available yet")
        return []

    all_rows = []
    for chunk_size in sizes:
        for chunk_overlap in overlaps:
            if chunk_overlap >= chunk_size:
                continue
            splitter = build_fixed_splitter(chunk_size, chunk_overlap)
            chunks = splitter.split_text(text)
            rows = summarize_chunks(chunks, label, chunk_size, chunk_overlap)
            all_rows.extend(rows)
            print("-")
    return all_rows

In [8]:
pdf_source_text = globals().get("pdf_normalized_text") or globals().get("pdf_text") or ""
podcast_source_text = globals().get("podcast_normalized_text") or globals().get("podcast_text") or ""

pdf_fixed_rows = run_fixed_size_experiments(pdf_source_text, "PDF")
podcast_fixed_rows = run_fixed_size_experiments(podcast_source_text, "Podcast")

fixed_rows = pdf_fixed_rows + podcast_fixed_rows
print(f"Total fixed-size chunk records: {len(fixed_rows)}")

PDF | chunk_size=500 | overlap=0
  chunks: 317
  min chars: 114
  max chars: 500
  avg chars: 497.6
  first chunk preview: INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI              ETHICS GUIDELINES  FOR  TRUSTWO
  last chunk preview: t deliverable , also contribut ed to the content of this document .     Nathalie Smuha provided editorial support.
-
PDF | chunk_size=500 | overlap=50
  chunks: 352
  min chars: 164
  max chars: 500
  avg chars: 497.8
  first chunk preview: INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI              ETHICS GUIDELINES  FOR  TRUSTWO
  last chunk preview: hair until 1 February 2019  coordinat ing the first deliverable , also contribut ed to the content of this document .     Nathalie Smuha provided editorial support.
-
PDF | chunk_size=500 | 

In [9]:
def preview_boundaries(rows, n=3):
    for row in rows[:n]:
        print(f"[{row['label']}] size={row['chunk_size']} overlap={row['chunk_overlap']} chunk={row['chunk_index']} chars={row['chunk_chars']}")
        text = row['chunk_text']
        print(text[:300].replace(chr(10), ' '))
        print('...')
        print(text[-300:].replace(chr(10), ' '))
        print()


print("PDF chunk previews:")
preview_boundaries(pdf_fixed_rows, n=3)

print("Podcast chunk previews:")
preview_boundaries(podcast_fixed_rows, n=3)

PDF chunk previews:
[PDF] size=500 overlap=0 chunk=1 chars=489
INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI              ETHICS GUIDELINES  FOR  TRUSTWORTHY  AI    High -Level Expert Group on Artificial Intelligence                   This document was 
...
OR  TRUSTWORTHY  AI    High -Level Expert Group on Artificial Intelligence                   This document was written by the High -Level Expert Group on AI (AI HLEG) . The members of the AI HLEG  named in this document support the overall framework for Trustworthy  AI put forward in these Guideline

[PDF] size=500 overlap=0 chunk=2 chars=500
s,  although they do not necessarily agree with every single statement in the document .     The Trustworthy  AI assessment  list presented in Chapter III of this document will undergo a piloting phase by  stakeholders to gather practical feedback. A revised version  of the assessment  

## Step 3 - Recursive chunking

This step uses a separator-aware splitter so the PDF can keep headings and numbered structure better, while the podcast transcript can stay closer to paragraph and sentence boundaries.

The recursive splitter is the first fair comparison against the fixed-size baseline.

In [10]:
import re

try:
    import pandas as pd
except ImportError:
    pd = None

from langchain_text_splitters import RecursiveCharacterTextSplitter


if "build_experiment_summary" not in globals():
    def sentence_break_flags(chunk: str):
        stripped = chunk.strip()
        if not stripped:
            return True, True
        first_char = stripped[0]
        starts_mid_sentence = first_char not in "[(\"'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        ends_mid_sentence = not re.search(r"[.!?][\"')\]]?\s*$", stripped)
        return starts_mid_sentence, ends_mid_sentence


    def build_experiment_summary(rows, original_text: str):
        summary_rows = []
        grouped = {}
        for row in rows:
            key = (row['label'], row['chunk_size'], row['chunk_overlap'])
            grouped.setdefault(key, []).append(row)

        for (label, chunk_size, chunk_overlap), group_rows in grouped.items():
            lengths = [row['chunk_chars'] for row in group_rows]
            starts_mid = 0
            ends_mid = 0
            for row in group_rows:
                starts_flag, ends_flag = sentence_break_flags(row['chunk_text'])
                starts_mid += int(starts_flag)
                ends_mid += int(ends_flag)

            chunk_total = len(group_rows)
            total_chars = len(original_text) if original_text else 0
            redundancy_pct = 0.0
            if total_chars:
                redundancy_pct = max(0.0, ((sum(lengths) - total_chars) / total_chars) * 100)

            summary_rows.append({
                "label": label,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "chunk_count": chunk_total,
                "avg_chars": round(sum(lengths) / chunk_total, 1),
                "min_chars": min(lengths),
                "max_chars": max(lengths),
                "start_mid_sentence_rate": round((starts_mid / chunk_total) * 100, 1),
                "end_mid_sentence_rate": round((ends_mid / chunk_total) * 100, 1),
                "redundancy_pct": round(redundancy_pct, 1),
            })

        return summary_rows


    def print_summary_table(summary_rows):
        if not summary_rows:
            print("No summary rows available")
            return

        ordered_rows = sorted(summary_rows, key=lambda item: (item["label"], item["chunk_size"], item["chunk_overlap"]))
        if pd is not None:
            print(pd.DataFrame(ordered_rows).to_string(index=False))
            return

        for row in ordered_rows:
            print(row)


if "preview_boundaries" not in globals():
    def preview_boundaries(rows, n=3):
        for row in rows[:n]:
            print(f"[{row['label']}] size={row['chunk_size']} overlap={row['chunk_overlap']} chunk={row['chunk_index']} chars={row['chunk_chars']}")
            text = row['chunk_text']
            print(text[:300].replace(chr(10), ' '))
            print('...')
            print(text[-300:].replace(chr(10), ' '))
            print()


def build_recursive_splitter(chunk_size: int, chunk_overlap: int, separators) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        separators=separators,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )


def run_recursive_experiments(text: str, label: str, separators, sizes=(500, 1000, 2000), overlaps=(0, 50, 100)):
    if not text:
        print(f"{label}: no text available yet")
        return []

    all_rows = []
    for chunk_size in sizes:
        for chunk_overlap in overlaps:
            if chunk_overlap >= chunk_size:
                continue
            splitter = build_recursive_splitter(chunk_size, chunk_overlap, separators)
            chunks = splitter.split_text(text)
            rows = summarize_chunks(chunks, label, chunk_size, chunk_overlap)
            all_rows.extend(rows)
            print("-")
    return all_rows


pdf_recursive_separators = ["\n\n", "\n", ". ", " ", ""]
podcast_recursive_separators = ["\n\n", "\n", ". ", " ", ""]

pdf_source_text = globals().get("pdf_normalized_text") or globals().get("pdf_text") or ""
podcast_source_text = globals().get("podcast_normalized_text") or globals().get("podcast_text") or ""

pdf_recursive_rows = run_recursive_experiments(
    pdf_source_text,
    "PDF recursive",
    pdf_recursive_separators,
)
podcast_recursive_rows = run_recursive_experiments(
    podcast_source_text,
    "Podcast recursive",
    podcast_recursive_separators,
)

pdf_recursive_summary = build_experiment_summary(pdf_recursive_rows, pdf_source_text)
podcast_recursive_summary = build_experiment_summary(podcast_recursive_rows, podcast_source_text)
recursive_summary = pdf_recursive_summary + podcast_recursive_summary

print("Recursive chunk comparison table:")
print_summary_table(recursive_summary)

print("PDF recursive chunk previews:")
preview_boundaries(pdf_recursive_rows, n=3)

print("Podcast recursive chunk previews:")
preview_boundaries(podcast_recursive_rows, n=3)


PDF recursive | chunk_size=500 | overlap=0
  chunks: 369
  min chars: 31
  max chars: 497
  avg chars: 423.1
  first chunk preview: INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI
  last chunk preview: iverable . Nozha  Boujemaa , Vice -Chair until 1 February 2019  coordinat ing the first deliverable , also contribut ed to the content of this document .     Nathalie Smuha provided editorial support.
-
PDF recursive | chunk_size=500 | overlap=50
  chunks: 372
  min chars: 31
  max chars: 497
  avg chars: 425.3
  first chunk preview: INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI
  last chunk preview: iverable . Nozha  Boujemaa , Vice -Chair until 1 February 2019  coordinat ing the first deliverable , also contribut ed to the content of this document .     Nathalie Smuha pro

## Step 4 - Token-based chunking

This step uses token counts instead of character counts so the comparison reflects model context more directly.

The token splitter should be the cleanest way to measure how the PDF and podcast behave under model-sized chunks.

In [11]:
import re

try:
    import pandas as pd
except ImportError:
    pd = None

import tiktoken
from langchain_text_splitters import TokenTextSplitter


token_encoding = tiktoken.get_encoding(os.getenv("TIKTOKEN_ENCODING", "cl100k_base"))


def token_count(text: str) -> int:
    return len(token_encoding.encode(text or ""))


def build_token_splitter(chunk_size: int, chunk_overlap: int) -> TokenTextSplitter:
    return TokenTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        encoding_name=token_encoding.name,
    )


def run_token_experiments(text: str, label: str, sizes=(500, 1000, 1500), overlaps=(0, 50, 100)):
    if not text:
        print(f"{label}: no text available yet")
        return []

    all_rows = []
    for chunk_size in sizes:
        for chunk_overlap in overlaps:
            if chunk_overlap >= chunk_size:
                continue
            splitter = build_token_splitter(chunk_size, chunk_overlap)
            chunks = splitter.split_text(text)
            if not chunks:
                print(f"{label} | chunk_size={chunk_size} | overlap={chunk_overlap} -> no chunks")
                continue

            rows = []
            print(f"{label} | chunk_size={chunk_size} | overlap={chunk_overlap}")
            print(f"  chunks: {len(chunks)}")
            token_lengths = [token_count(chunk) for chunk in chunks]
            char_lengths = [len(chunk) for chunk in chunks]
            print(f"  min tokens: {min(token_lengths)}")
            print(f"  max tokens: {max(token_lengths)}")
            print(f"  avg tokens: {sum(token_lengths) / len(token_lengths):.1f}")
            print(f"  min chars: {min(char_lengths)}")
            print(f"  max chars: {max(char_lengths)}")
            print(f"  avg chars: {sum(char_lengths) / len(char_lengths):.1f}")
            print(f"  first chunk preview: {chunks[0][:200].replace(chr(10), ' ')}")
            print(f"  last chunk preview: {chunks[-1][-200:].replace(chr(10), ' ')}")

            for i, chunk in enumerate(chunks):
                rows.append({
                    "label": label,
                    "chunk_size": chunk_size,
                    "chunk_overlap": chunk_overlap,
                    "chunk_index": i + 1,
                    "chunk_tokens": token_count(chunk),
                    "chunk_chars": len(chunk),
                    "chunk_text": chunk,
                })
            all_rows.extend(rows)
            print("-")
    return all_rows


def build_token_summary(rows, original_text: str):
    if not rows:
        return []

    summary_rows = []
    grouped = {}
    for row in rows:
        key = (row['label'], row['chunk_size'], row['chunk_overlap'])
        grouped.setdefault(key, []).append(row)

    for (label, chunk_size, chunk_overlap), group_rows in grouped.items():
        token_lengths = [row['chunk_tokens'] for row in group_rows]
        char_lengths = [row['chunk_chars'] for row in group_rows]
        summary_rows.append({
            "label": label,
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "chunk_count": len(group_rows),
            "avg_tokens": round(sum(token_lengths) / len(group_rows), 1),
            "min_tokens": min(token_lengths),
            "max_tokens": max(token_lengths),
            "avg_chars": round(sum(char_lengths) / len(group_rows), 1),
            "min_chars": min(char_lengths),
            "max_chars": max(char_lengths),
            "redundancy_pct": round(max(0.0, ((sum(char_lengths) - len(original_text)) / len(original_text)) * 100) if original_text else 0.0, 1),
        })

    return summary_rows


def print_token_summary(summary_rows):
    if not summary_rows:
        print("No token summary rows available")
        return
    ordered_rows = sorted(summary_rows, key=lambda item: (item["label"], item["chunk_size"], item["chunk_overlap"]))
    if pd is not None:
        print(pd.DataFrame(ordered_rows).to_string(index=False))
        return
    for row in ordered_rows:
        print(row)


pdf_source_text = globals().get("pdf_source_text") or globals().get("pdf_normalized_text") or globals().get("pdf_text") or ""
podcast_source_text = globals().get("podcast_source_text") or globals().get("podcast_normalized_text") or globals().get("podcast_text") or ""

pdf_token_rows = run_token_experiments(pdf_source_text, "PDF token")
podcast_token_rows = run_token_experiments(podcast_source_text, "Podcast token")

pdf_token_summary = build_token_summary(pdf_token_rows, pdf_source_text)
podcast_token_summary = build_token_summary(podcast_token_rows, podcast_source_text)
token_summary = pdf_token_summary + podcast_token_summary

print("Token chunk comparison table:")
print_token_summary(token_summary)

print("PDF token chunk previews:")
preview_boundaries(pdf_token_rows, n=3)

print("Podcast token chunk previews:")
preview_boundaries(podcast_token_rows, n=3)


PDF token | chunk_size=500 | overlap=0
  chunks: 66
  min tokens: 366
  max tokens: 500
  avg tokens: 498.0
  min chars: 1241
  max chars: 2701
  avg chars: 2395.7
  first chunk preview:            INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI              ETHICS GUIDELINES  F
  last chunk preview: able . Nozha  Boujemaa , Vice -Chair until 1 February 2019  coordinat ing the first deliverable , also contribut ed to the content of this document .     Nathalie Smuha provided editorial support.    
-
PDF token | chunk_size=500 | overlap=50
  chunks: 73
  min tokens: 466
  max tokens: 500
  avg tokens: 499.5
  min chars: 1635
  max chars: 2701
  avg chars: 2402.0
  first chunk preview:            INDEPENDENT   HIGH-LEVEL EXPERT GROUP ON   ARTIFICIAL INTELLIGENCE   SET UP BY THE EUROPEAN COMMISSION               ETHICS GUIDELINES   FOR TRUSTWORTHY AI              ETHICS GUIDELINES  

## Step 5 - Recommendation and wrap-up

This final step turns the measured chunking results into a clear recommendation for each source.

The goal is to state which strategy is strongest for the PDF and which is most sensible for the podcast transcript, using the outputs from the earlier steps.

In [12]:
import re

try:
    import pandas as pd
except ImportError:
    pd = None


def pick_pdf_row(summary_rows):
    if not summary_rows:
        return None
    preferred = [row for row in summary_rows if row["chunk_size"] == 1000]
    if preferred:
        return sorted(preferred, key=lambda row: (row["redundancy_pct"], row["chunk_overlap"]))[0]
    return sorted(summary_rows, key=lambda row: (abs(row["chunk_size"] - 1000), row["redundancy_pct"], row["chunk_overlap"]))[0]


def pick_podcast_row(summary_rows):
    if not summary_rows:
        return None

    recursive_rows = [row for row in summary_rows if row["label"].startswith("Podcast recursive")]
    if recursive_rows:
        preferred = [row for row in recursive_rows if row["chunk_size"] == 1500 and row["chunk_overlap"] == 0]
        if preferred:
            return preferred[0]
        preferred = [row for row in recursive_rows if row["chunk_size"] == 1000]
        if preferred:
            return sorted(preferred, key=lambda row: (row["chunk_overlap"], row["chunk_count"]))[0]
        return sorted(recursive_rows, key=lambda row: (row["chunk_size"], row["chunk_overlap"]))[0]

    return sorted(summary_rows, key=lambda row: (row["chunk_count"], row["chunk_size"], row["chunk_overlap"]))[0]


pdf_token_summary_rows = globals().get("pdf_token_summary", [])
podcast_token_summary_rows = globals().get("podcast_token_summary", [])
pdf_recursive_summary_rows = globals().get("pdf_recursive_summary", [])
podcast_recursive_summary_rows = globals().get("podcast_recursive_summary", [])
fixed_summary_rows = globals().get("fixed_summary", [])

pdf_recommended = pick_pdf_row(pdf_token_summary_rows)
podcast_recommended = pick_podcast_row(podcast_recursive_summary_rows or podcast_token_summary_rows)

recommendation_rows = []
if pdf_recommended:
    recommendation_rows.append({
        "source": "PDF",
        "strategy": pdf_recommended["label"],
        "chunk_size": pdf_recommended["chunk_size"],
        "overlap": pdf_recommended["chunk_overlap"],
        "reason": "Lowest redundancy among the token-based PDF runs while keeping chunk size near the model boundary.",
    })
if podcast_recommended:
    recommendation_rows.append({
        "source": "Podcast",
        "strategy": podcast_recommended["label"],
        "chunk_size": podcast_recommended["chunk_size"],
        "overlap": podcast_recommended["chunk_overlap"],
        "reason": "The transcript is short enough that the best result is the smallest practical split with no unnecessary overlap.",
    })

print("Final recommendation table:")
if pd is not None and recommendation_rows:
    print(pd.DataFrame(recommendation_rows).to_string(index=False))
else:
    for row in recommendation_rows:
        print(row)

print("\nShort conclusion:")
print("- The PDF benefits most from token-based chunking, because it keeps chunk sizes aligned with model context and reduces arbitrary character cuts.")
print("- The podcast transcript is now a real multi-part transcript, so the better choice is recursive chunking with a moderate chunk size that preserves conversational boundaries without exploding the chunk count.")
print("- For submission, keep the notebook outputs, the real transcript files in outputs/, and a brief README/lab_summary at the repo root.")


Final recommendation table:
 source          strategy  chunk_size  overlap                                                                                                           reason
    PDF         PDF token        1000        0               Lowest redundancy among the token-based PDF runs while keeping chunk size near the model boundary.
Podcast Podcast recursive        1000        0 The transcript is short enough that the best result is the smallest practical split with no unnecessary overlap.

Short conclusion:
- The PDF benefits most from token-based chunking, because it keeps chunk sizes aligned with model context and reduces arbitrary character cuts.
- The podcast transcript is now a real multi-part transcript, so the better choice is recursive chunking with a moderate chunk size that preserves conversational boundaries without exploding the chunk count.
- For submission, keep the notebook outputs, the real transcript files in outputs/, and a brief README/lab_summary at th